In [2]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/datasets/khadejabk/lab08-llm/lab8_llm.py


In [3]:
import glob
import shutil
import os

files = glob.glob("/kaggle/input/**/lab8_llm.py", recursive=True)

print("Found:", files)

shutil.copy(files[0], "/kaggle/working/lab8_llm.py")

print("File copied successfully!")

Found: ['/kaggle/input/datasets/khadejabk/lab08-llm/lab8_llm.py']
File copied successfully!


In [4]:
import lab8_llm as llm
from lab8_llm import ask

llm.set_seed(0)

print(ask("Say hello in one short sentence."))

Hello from the Lab 8 stand-in. I am a Python function, not a language model.


In [5]:
call = llm.LOG[-1]

print("calls so far:", len(llm.LOG),
      "| seconds:", call["seconds"],
      "| reply length:", len(call["reply"]))

calls so far: 1 | seconds: 0.0 | reply length: 76


In [7]:
QUESTION = "Why does the game crash when the player opens the inventory?"

CONTEXT = {
    "log": "Error log: " + llm.LOG_TEXT.replace("\n", " "),
    "script": "Script line: 42: label.text = slot.item.name;",
    "version": "Engine version: Unity 2022.3 LTS"
}

def build_prompt(question, items):
    parts = [CONTEXT[name] for name in items]
    return "\n".join(parts + ["Question: " + question])

In [8]:
for items in [[], ["log"]]:
    prompt = build_prompt(QUESTION, items)
    reply = ask(prompt)

    print(f"context {items or 'none'}: {len(prompt)} characters sent")
    print(" reply:", reply)

context none: 70 characters sent
 reply: I cannot be sure without more information. Common causes of a crash when a menu opens are null references, missing assets or endless loops.
context ['log']: 223 characters sent
 reply: Likely cause: a NullReferenceException at InventoryUI.cs:42, so something on that line is empty.


In [10]:
def clean_json(text):
    """Remove a ```json fence if the model added one."""
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1].rsplit("```", 1)[0]
    return text.strip()


def check_plan(text):
    """Check whether the response is a valid plan."""
    try:
        plan = json.loads(clean_json(text))
    except json.JSONDecodeError:
        return None, "not valid JSON"

    if not isinstance(plan, list) or not 3 <= len(plan) <= 6:
        return None, "needs 3 to 6 steps"

    if not all(isinstance(s, dict) and {"step", "check"} <= s.keys() for s in plan):
        return None, "a step is missing 'step' or 'check'"

    return plan, ""


def plan_prompt(goal):
    return (f"Goal: {goal}\nSplit the goal into 3 to 6 steps. Reply with a JSON array only. "
            'Each item must look like {"step": "...", "check": "how to know it is done"}.')

In [11]:
for seed in (0, 1):
    llm.set_seed(seed)

    plan, problem = check_plan(
        ask(plan_prompt("Create a simple enemy AI prototype"))
    )

    print(f"seed {seed}: problem = {problem or 'none'}")

    for i, item in enumerate(plan or [], 1):
        print(f" {i}. {item['step']} -> check: {item['check']}")

seed 0: problem = none
 1. Define behaviour -> check: Patrol, chase and retreat are written down
 2. Choose states -> check: Each behaviour is a named state
 3. Implement logic -> check: Events switch the enemy between states
 4. Test scenarios -> check: Player distance and obstacles are tested
seed 1: problem = not valid JSON


In [12]:
REPORTS = [
    ("Player falls through the floor after a jump", "Physics / Collision"),
    ("Enemy walks through a wall", "Physics / Collision"),
    ("Game freezes for five seconds when loading level 3", "Performance"),
    ("Frame rate drops to 12 fps in the forest scene", "Performance"),
    ("Textures take a long time to appear", "Performance"),
    ("Boss health bar is not shown on the screen", "UI"),
    ("Inventory button does not respond when clicked", "UI"),
    ("Score resets to zero after pausing", "Game Logic"),
    ("Enemy keeps patrolling after the player is dead", "Game Logic"),
    ("Door stays closed even after the key is collected", "Game Logic"),
    ("Footstep sound plays twice for each step", "Audio"),
    ("Music stops when the menu is opened", "Audio"),
]

In [21]:
CATEGORIES = "Physics / Collision, Performance, UI, Game Logic, Audio"

ZERO = ("Classify this bug report as one of: " + CATEGORIES + ". "
        "Also give a severity (Low, Medium or High) and one action.\nReport: {report}")

FEW = ("Classify bug reports. Use exactly the format: Category | Severity | Action\n"
       "Input: Player falls through the floor\n"
       "Output: Physics / Collision | High | Check collider and collision layer\n"
       "Input: Loading screen freezes the game\n"
       "Output: Performance | Medium | Profile the scene and check asset sizes\n"
       "Input: {report}\nOutput:")

import re

PATTERN = re.compile(
    r"^(Physics / Collision|Performance|UI|Game Logic|Audio) \| (Low|Medium|High) \| (.+)$"
)

def parse(reply):
    m = PATTERN.match(reply.strip())
    return (m.group(1), m.group(2), m.group(3)) if m else None

In [20]:
import re

PATTERN = re.compile(
    r"^(Physics / Collision|Performance|UI|Game Logic|Audio) \| (Low|Medium|High) \| (.+)$"
)

def parse(reply):
    match = PATTERN.match(reply.strip())
    if not match:
        return None
    return match.groups()

In [23]:
def run_method(template, seed):
    """One full pass over the 12 reports."""
    llm.set_seed(seed)
    valid = correct = 0

    for report, truth in REPORTS:
        parsed = parse(ask(template.format(report=report)))

        if parsed:
            valid += 1
            correct += parsed[0] == truth

    return valid, correct

In [24]:
for name, template in [("zero-shot", ZERO), ("few-shot", FEW)]:
    results = [run_method(template, seed) for seed in range(10)]

    valid = sum(v for v, _ in results)
    correct = sum(c for _, c in results)
    total = 10 * len(REPORTS)

    print(f"{name}: format valid {valid}/{total} ({valid / total:.0%}), "
          f"correct among valid {correct}/{valid} ({correct / valid:.0%}), "
          f"correct overall {correct}/{total} ({correct / total:.0%})")

zero-shot: format valid 49/120 (41%), correct among valid 34/49 (69%), correct overall 34/120 (28%)
few-shot: format valid 115/120 (96%), correct among valid 107/115 (93%), correct overall 107/120 (89%)


In [25]:
def score_answer(reply):
    """0 = no cause, 1 = names the exception, 2 = names the exception and slot.item."""
    reply = reply.lower()

    if "nullreferenceexception" in reply and "slot.item" in reply:
        return 2
    elif "nullreferenceexception" in reply:
        return 1
    else:
        return 0

In [26]:
for items in [[], ["log"], ["log", "script"], ["log", "script", "version"]]:
    prompt = build_prompt(QUESTION, items)
    llm.set_seed(0)

    reply = ask(prompt)
    score = score_answer(reply)

    print(f"context {items or 'none'} characters {len(prompt)} score {score}")

context none characters 70 score 0
context ['log'] characters 223 score 1
context ['log', 'script'] characters 269 score 2
context ['log', 'script', 'version'] characters 302 score 2


In [29]:
def get_plan(goal, max_retries=2):
    """Return (plan, attempts_used). plan is None if every attempt failed."""
    prompt = plan_prompt(goal)

    for attempt in range(max_retries + 1):
        reply = ask(prompt)
        plan, problem = check_plan(reply)

        if plan is not None:
            return plan, attempt + 1

        prompt += (
            "\nYour previous reply was rejected because: "
            + str(problem)
            + ". Please correct the problem and return valid JSON."
        )

    return None, max_retries + 1

In [30]:
GOALS = [
    "Create a simple enemy AI prototype",
    "Prepare a playable demo for Friday",
    "Add a save system to the game"
]

first_try_ok = retries_used = failures = 0

llm.LOG.clear()

for seed in range(20):
    llm.set_seed(seed)

    for goal in GOALS:
        plan, attempts = get_plan(goal)

        if plan is None:
            failures += 1
        elif attempts == 1:
            first_try_ok += 1
        else:
            retries_used += attempts - 1

In [31]:
total = 20 * len(GOALS)

print(f"plans asked for: {total}")
print(f"valid on the first try: {first_try_ok} ({first_try_ok / total:.0%})")
print(f"retries used: {retries_used}")
print(f"gave up after retries: {failures}")
print(f"model calls in total: {len(llm.LOG)}")

plans asked for: 60
valid on the first try: 40 (67%)
retries used: 22
gave up after retries: 0
model calls in total: 82


In [32]:
FILES = {
    "InventoryUI.cs": "40: foreach (var slot in slots)\n41: {\n42:     label.text = slot.item.name;\n43: }"
}

TOOLS = {
    "read_log": lambda arg: llm.LOG_TEXT,
    "search_file": lambda arg: FILES.get(arg, "file not found: " + arg),
    "check_version": lambda arg: "Unity 2022.3 LTS",
}

HEADER = (
    "You are a debugging agent. Available tools: read_log[], search_file[name], check_version[].\n"
    "Reply with 'Thought: ...' and then either 'Action: tool[argument]' or 'Final: answer'.\n"
)

def parse_step(reply):
    """Return ('final', text) or ('action', tool, arg) or ('bad', reply)."""
    final = re.search(r"Final:\s*(.+)", reply, re.S)
    if final:
        return ("final", final.group(1).strip())

    act = re.search(r"Action:\s*(\w+)\[(.*?)\]", reply)
    if act:
        return ("action", act.group(1), act.group(2))

    return ("bad", reply)

In [33]:
def react(question, max_steps=5):
    transcript = HEADER + "Question: " + question + "\n"
    trace = []

    for step in range(1, max_steps + 1):
        reply = ask(transcript)
        kind = parse_step(reply)

        if kind[0] == "final":
            trace.append(f"step {step}: Final -> {kind[1]}")
            return kind[1], trace

        elif kind[0] == "bad":
            observation = "Please use the required format: Thought: ... followed by Action: tool[argument] or Final: answer."

        elif kind[0] == "action":
            tool, arg = kind[1], kind[2]

            if tool not in TOOLS:
                observation = (
                    f"unknown tool {tool}. Allowed: "
                    + ", ".join(TOOLS.keys())
                )
            else:
                result = TOOLS[tool](arg)
                observation = str(result)

        trace.append(f"step {step}: {kind[1] if kind[0] == 'action' else kind[0]} -> {observation[:100]}")
        transcript += reply + "\nObservation: " + observation + "\n"

    return None, trace + ["stopped: step limit reached"]

In [34]:
QUESTION4 = "Why does the game crash when the inventory opens?"

llm.set_seed(2)
answer, trace = react(QUESTION4)

print("\n".join(trace))
print("answer:", answer)

outcomes = {"solved": 0, "step limit": 0}
unknown_tool_seen = 0

for seed in range(30):
    llm.set_seed(seed)
    llm.LOG.clear()

    answer, trace = react(QUESTION4)

    outcomes["solved" if answer else "step limit"] += 1
    unknown_tool_seen += any("unknown tool" in t for t in trace)

print("30 normal runs:", outcomes,
      "| runs that tried an unknown tool:", unknown_tool_seen)

llm.set_seed(0, mode="stuck")
answer, trace = react(QUESTION4)

print("stuck model:", answer, "|", trace[-1],
      "| steps:", len(trace) - 1)

step 1: read_log -> ERROR NullReferenceException: Object reference not set to an instance of an object
  at InventoryUI.
step 2: search_file -> 40: foreach (var slot in slots)
41: {
42:     label.text = slot.item.name;
43: }
step 3: run_game -> unknown tool run_game. Allowed: read_log, search_file, check_version
step 4: run_game -> unknown tool run_game. Allowed: read_log, search_file, check_version
step 5: Final -> In InventoryUI.cs line 42, check that slot.item is not null before reading name.
answer: In InventoryUI.cs line 42, check that slot.item is not null before reading name.
30 normal runs: {'solved': 30, 'step limit': 0} | runs that tried an unknown tool: 3
stuck model: None | stopped: step limit reached | steps: 5


In [38]:
import re

TRUSTED = {
    "Physics.Raycast",
    "transform.Translate",
    "Destroy",
    "audioSource.Play",
    "transform.Find"
}

TASKS = [
    "cast a ray forward from the player",
    "move the character with its speed",
    "destroy the object after two seconds",
    "play the sound attached to this object",
    "find the child object called Weapon"
]

def called_names(line):
    return re.findall(r"([A-Za-z_]\w*(?:\.[A-Za-z_]\w*)*)\s*\(", line)

def unsupported(line):
    """Return the called names in the line that are not in TRUSTED."""
    return [name for name in called_names(line) if name not in TRUSTED]

In [39]:
llm.set_seed(3)

for task in TASKS:
    line = ask(f"Write one line of C# to {task}.")
    bad = unsupported(line)

    if bad:
        print("UNSUPPORTED", bad, "|", line)
    else:
        print("supported    |", line)

UNSUPPORTED ['Physics.RaycastSmart'] | Physics.RaycastSmart(origin, direction);
supported    | transform.Translate(move * speed * Time.deltaTime);
supported    | Destroy(gameObject, 2f);
supported    | audioSource.Play();
supported    | transform.Find("Weapon");


In [40]:
flagged = 0
wrong_destroy = 0
total = 0

for seed in range(40):
    llm.set_seed(seed)

    for task in TASKS:
        line = ask(f"Write one line of C# to {task}.")
        total += 1

        if unsupported(line):
            flagged += 1

        if line.strip() == "Destroy(2f, gameObject);":
            wrong_destroy += 1

print(f"lines checked: {total}, flagged as unsupported: {flagged} ({flagged / total:.1%})")
print(f"real function used wrongly and NOT flagged: {wrong_destroy}")

lines checked: 200, flagged as unsupported: 54 (27.0%)
real function used wrongly and NOT flagged: 2


In [41]:
total_calls = 0
solved = 0

for seed in range(30):
    llm.set_seed(seed)
    llm.LOG.clear()

    answer, trace = react(QUESTION4)

    total_calls += len(llm.LOG)

    if answer:
        solved += 1

print("Total model calls:", total_calls)
print("Solved:", solved, "/ 30")
print("Average calls per answer:", round(total_calls / 30, 2))

Total model calls: 94
Solved: 30 / 30
Average calls per answer: 3.13


In [42]:

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

CATEGORIES_LIST = [
    "Physics / Collision",
    "Performance",
    "UI",
    "Game Logic",
    "Audio"
]

def evaluate_method(template):
    y_true = []
    y_pred = []

    for seed in range(10):
        llm.set_seed(seed)

        for report, correct_category in REPORTS:
            reply = ask(template.format(report=report))
            parsed = parse(reply)

            y_true.append(correct_category)

            # Count invalid responses as incorrect predictions
            if parsed:
                y_pred.append(parsed[0])
            else:
                y_pred.append("Invalid")

    accuracy = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true, y_pred,
        labels=CATEGORIES_LIST,
        average="macro",
        zero_division=0
    )

    recall = recall_score(
        y_true, y_pred,
        labels=CATEGORIES_LIST,
        average="macro",
        zero_division=0
    )

    f1 = f1_score(
        y_true, y_pred,
        labels=CATEGORIES_LIST,
        average="macro",
        zero_division=0
    )

    return accuracy, precision, recall, f1


for name, template in [("Zero-shot", ZERO), ("Few-shot", FEW)]:
    accuracy, precision, recall, f1 = evaluate_method(template)

    print("\n", name)
    print("Accuracy: ", round(accuracy * 100, 2), "%")
    print("Precision:", round(precision * 100, 2), "%")
    print("Recall:   ", round(recall * 100, 2), "%")
    print("F1 Score: ", round(f1 * 100, 2), "%")



 Zero-shot
Accuracy:  28.33 %
Precision: 69.12 %
Recall:    28.0 %
F1 Score:  39.81 %

 Few-shot
Accuracy:  89.17 %
Precision: 92.8 %
Recall:    90.33 %
F1 Score:  91.1 %
